In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.svm import SVC
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.model_selection import train_test_split,GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import LabelEncoder
from sklearn.pipeline import Pipeline





In [ ]:
data=pd.read_csv(r"E:\FCDS\Semester 6\Data Computation\Projects\star_classification.csv")

In [ ]:
data.head()

In [ ]:
data.shape

In [ ]:
data.info()

------------------------------------------------------------------------------------
Data Cleaning
---------------------------------------------------------------------------------------


-------------------------------------------------------
Remove duplicated
---------------------------------------------------------------

In [ ]:
data.duplicated().sum()

In [ ]:
data.drop_duplicates(inplace=True)

In [ ]:
data.duplicated().sum()

In [ ]:
data.nunique() == 1 #rerun_ID ==>    True


In [ ]:
data.drop("rerun_ID", axis=1,inplace=True)

In [ ]:
data.drop("obj_ID",axis=1,inplace=True)

In [ ]:
data.head(10)

In [ ]:
len(data["run_ID"].unique())

In [ ]:
len(data["field_ID"].unique())

In [ ]:
len(data["spec_obj_ID"].unique())

----------------------------------------------------------------------------------------------
Fill NULL
-------------------

In [ ]:
data.isnull().sum()

In [ ]:
means = data.mean(numeric_only=True)
data = data.fillna(means)

------------------------------------
Remove Outliers
------------------------------------------

In [ ]:
numeric_cols = data.select_dtypes(include='number').columns
plt.figure(figsize=(12,6))

plt.boxplot([data[col] for col in numeric_cols], labels=numeric_cols)

plt.title("Before Outlier Treatment (All Columns)")
plt.xticks(rotation=45)

plt.show()

In [ ]:
numeric_cols = data.select_dtypes(include='number').columns

for col in numeric_cols:
    Q1 = data[col].quantile(0.25)
    Q3 = data[col].quantile(0.75)
    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    data[col] = data[col].clip(lower, upper)    #outlierبيشيل الصف اللي جواه 
    print(col, len(data))

In [ ]:
numeric_cols = data.select_dtypes(include='number').columns

plt.figure(figsize=(12,6))

plt.boxplot([data[col] for col in numeric_cols], labels=numeric_cols)

plt.title("After Outlier Treatment (All Columns)")
plt.xticks(rotation=45)

plt.show()

--------------------------------------------------------

In [ ]:

le = LabelEncoder()
data['class'] = le.fit_transform(data['class'])

In [ ]:
plt.figure(figsize=(10, 8))
cor = data.corr()
sns.heatmap(cor, annot=True, cmap="coolwarm", linewidths=0.5)
plt.show()

In [ ]:
data.drop("MJD",axis=1,inplace=True) #0.0004 corr with target "class"

-------------------------------------
Split Data into Train Test 
-------------------------------------


In [ ]:
train, test= train_test_split(
    data,
    test_size=0.2,
    random_state=1)
train.shape, test.shape

In [ ]:
x_train = train.drop("class", axis=1)
y_train=train.loc[:,"class"]
x_test= test.drop("class", axis=1)
y_test=test.loc[:,"class"]


-------------------------------------
Normalization & PCA & LDA & SVM
----------------------


In [ ]:
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ("lda",LDA()),
    ('svc', SVC())
])

In [ ]:
param_grid = {
    'lda__n_components': [1,2],
    'svc__C': [0.1, 1, 10],
    'svc__kernel': ['rbf'],
    'svc__gamma': ['scale', 'auto']
}

In [ ]:
grid = GridSearchCV(
    pipe,
    param_grid,
    cv=3,
    scoring='accuracy',
    n_jobs=-1,
    verbose=2
)

grid.fit(x_train, y_train)

In [ ]:
print("Best Params:", grid.best_params_)
print("Best CV Score:", grid.best_score_)

In [ ]:
best_model = grid.best_estimator_

y_pred = best_model.predict(x_test)

print("Test Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

In [ ]:
plt.scatter(x_train[:,0], x_train[:,1], c=y_train, cmap='viridis', s=3)

ax = plt.gca()
xlim = ax.get_xlim()
ylim = ax.get_ylim()

xx = np.linspace(xlim[0], xlim[1], 100)
yy = np.linspace(ylim[0], ylim[1], 100)
YY, XX = np.meshgrid(yy, xx)

xy = np.c_[XX.ravel(), YY.ravel()]
Z = best_model.predict(xy)
Z = Z.reshape(XX.shape)

ax.contourf(XX, YY, Z, alpha=0.3)

plt.title("SVM Decision Boundary")
plt.xlabel("LDA Component 1")
plt.ylabel("LDA Component 2")
plt.show()